In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

In [ ]:
data = pd.read_csv("Final_Data.csv", index_col="Date")
data

In [ ]:
df = data.fillna(method="ffill")

### Define numeric features 

In [ ]:

numeric_features = df.select_dtypes(include=['number']).columns

selector = VarianceThreshold(threshold=0.5)  
filtered = selector.fit_transform(df[numeric_features])
selected_columns = df[numeric_features].columns[selector.get_support()]

print(selected_columns)

### Create lag features safely

In [ ]:
df.loc[:, 'Temperature_Lag1'] = df['Temperature at 2M(C)'].shift(1)
df.loc[:, 'Temperature_Lag2'] = df['Temperature at 2M(C)'].shift(2)
df.loc[:, 'Temperature_Lag3'] = df['Temperature at 2M(C)'].shift(3)

### Create rolling average features safely

In [ ]:
df.loc[:, 'Temperature_Rolling7'] = df['Temperature at 2M(C)'].rolling(window=7).mean()
df.loc[:, 'Temperature_Rolling30'] = df['Temperature at 2M(C)'].rolling(window=30).mean()
# Drop rows with NaN values (due to lagging and rolling)
df = df.dropna().reset_index(drop=True)
# Check the new columns
print(df[['Temperature at 2M(C)', 'Temperature_Lag1', 'Temperature_Lag2', 'Temperature_Lag3', 'Temperature_Rolling7']].head())

# Daily weather prediciton model

In [ ]:
# Define X and y
X = df[selected_columns]    # Features
y = df['Temperature at 2M(C)']  # Target to predict (daily temp)

# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train Random Forest

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train_scaled, y_train)

### Predict

In [ ]:
y_pred = model.predict(X_test_scaled)

### Evaluate

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R? Score: {r2:.2f}")

In [ ]:
# Select the latest row of those features from your dataframe
latest_data = df[selected_columns].iloc[-1:]

# Scale the input using the trained scaler
latest_data_scaled = scaler.transform(latest_data)

# Predict the temperature
forecast = model.predict(latest_data_scaled)

print(f"Predicted Temperature for Tomorrow: {forecast[0]:.2f} ?C")


###  Plot Actual vs Predicted

In [ ]:

plt.figure(figsize=(12,6))
plt.plot(y_test.values, label='Actual Temperature')
plt.plot(y_pred, label='Predicted Temperature')
plt.legend()
plt.title('Actual vs Predicted Daily Temperature')
plt.show()

### Define Features and Multiple Targets

In [ ]:
if df.select_dtypes(include='object').shape[1] > 0:
    df = pd.get_dummies(df)  # One-hot encode categorical columns

# Split Features and Targets
X = df.drop(columns=['Temperature at 2M(C)', 'Relative Humidity at 2 M(%)', 
                     'Precipitation Corrected (mm/day)', 'Wind Speed at 10 M Max (m/s)'])

y = df[['Temperature at 2M(C)', 'Relative Humidity at 2 M(%)', 
        'Precipitation Corrected (mm/day)', 'Wind Speed at 10 M Max (m/s)']]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Multi-Output Model

In [ ]:
base_model = RandomForestRegressor(n_estimators=100, random_state=42)
multi_model = MultiOutputRegressor(base_model)

multi_model.fit(X_train_scaled, y_train)

### Predictions

In [ ]:
y_pred = multi_model.predict(X_test_scaled)

### Evaluation

In [ ]:

for idx, col in enumerate(y.columns):
    print(f"--- {col} ---")
    print("RMSE:", np.sqrt(mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])))
    print("R2 Score:", r2_score(y_test.iloc[:, idx], y_pred[:, idx]))
    print()


### weather classification

In [ ]:

def classify_weather(temperature, humidity, precipitation, wind_speed, dew_point): 
    if temperature < 0:
        # Freezing temperatures, likely snow
        return "Snowy"
    elif 0 <= temperature < 10:
        # Cold weather, could be cloudy or overcast
        if precipitation > 1:  # Some precipitation
            return "Rainy"
        elif wind_speed > 5:  # High winds
            return "Windy"
        else:
            return "Cloudy"
    elif 10 <= temperature < 20:
        # Mild weather, partly cloudy or sunny
        if precipitation > 1:
            return "Rainy"
        elif humidity > 80:  # High humidity
            return "Foggy"
        else:
            return "Partly Cloudy"
    elif 20 <= temperature < 30:
        # Warm, likely sunny
        if humidity > 60:  # Higher humidity
            return "Humid"
        elif wind_speed > 5:
            return "Windy"
        else:
            return "Sunny"
    else:
        # Hot weather, possibly thunderstorms
        if precipitation > 5:
            return "Thunderstorm"
        elif humidity > 70:
            return "Humid"
        else:
            return "Sunny"

### Example usage

In [ ]:

predicted_temperature = 35  # Example temperature
humidity = 50  # Example humidity in percentage
precipitation = 0.5  # Example precipitation in mm/day
wind_speed = 5.0  # Example wind speed in m/s
dew_point = 5.0  # Example dew point in Celsius

weather_type = classify_weather(predicted_temperature, humidity, precipitation, wind_speed, dew_point)
print(f"Predicted Weather: {weather_type}")


#

#

# Extreme weather prediction

In [ ]:
# Define extreme weather conditions
selected_features = [
    'Temperature at 2M(C)',
    'Precipitation Corrected (mm/day)',
    'Wind Speed at 10 M (m/s)',
    'Relative Humidity at 2 M(%)'
]

df['Extreme_Weather'] = 0

# High temperature (>35?C)
df.loc[df['Temperature at 2M(C)'] > 35, 'Extreme_Weather'] = 1

# Heavy precipitation (>50mm/day)
df.loc[df['Precipitation Corrected (mm/day)'] > 50, 'Extreme_Weather'] = 1

# High wind speed (>20 m/s)
df.loc[df['Wind Speed at 10 M (m/s)'] > 20, 'Extreme_Weather'] = 1

# High humidity (>90%)
df.loc[df['Relative Humidity at 2 M(%)'] > 90, 'Extreme_Weather'] = 1


In [ ]:
X = df[selected_features]
y = df['Extreme_Weather']

### Split data into training and testing sets

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Initialize the model

In [ ]:

logreg = LogisticRegression(max_iter=1000)

### Train the model

In [ ]:

logreg.fit(X_train, y_train)


## Predict model

In [ ]:
y_pred = logreg.predict(X_test)

### Evaluate the model

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

#

#

# seasonality predictions

In [ ]:

df = pd.read_csv('Final_Data.csv', index_col = 'Date')
selected_columns = ['Year', 'Month', 'Day', 'Temperature at 2M(C)', '2Dew/Frost 2M(C)', 
                    'Earth Skin Temperature (C)', 'Temperature at 2M Min (C)', 
                    'Temperature at 2M Range', 'Temperature 2M Max(C)', 
                    'Specific Humidity at 2 M (g/kg)', 'Relative Humidity at 2 M(%)', 
                    'Precipitation Corrected (mm/day)', 'Wind Speed at 10 M Max (m/s)', 
                    'Wind Direction at 10M (Degrees)', 'Sky Surface Shortwave Downward Irradiance',
                    'Temperature at 2M(C)_7d_avg', 'Temperature at 2M(C)_30d_avg', 
                    'Temperature at 2M(C)_lag1', 'Relative Humidity at 2 M(%)_7d_avg', 
                    'Relative Humidity at 2 M(%)_30d_avg', 'Relative Humidity at 2 M(%)_lag1', 
                    'Feels_Like_Temp', 'Daily_Temp_Range', 'Dew_Point', 'Wind_U','Season', 
                    'Wind_V', 'Wind_Gust_Diff']


### Prepare features (X) and target (y)

In [ ]:
# Check if 'Season' column has all classes
print("Class distribution in dataset:")
print(df['Season'].value_counts())

# Split features and target
X = df[selected_columns].drop('Season', axis=1)
y = df['Season']

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

### Split data into train and test

In [ ]:
# Train-test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

### Scale features 


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



### Build and train the XGBoost model

In [ ]:

model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective='multi:softmax',  # Since we have multiple classes
    num_class=4  # Spring, Summer, Autumn, Winter
)

model.fit(X_train_scaled, y_train)

### Predict and evaluate

In [ ]:

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n? Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))